# Data Quality Filtering for Embedding Training

[![Open In Colab](https://img.shields.io/badge/Open%20In-Colab-blue?style=for-the-badge&logo=google-colab)](https://colab.research.google.com/github/dnth/rag-datakit/blob/main/nbs/03_filtering-quality-data.ipynb)
[![Open In Kaggle](https://img.shields.io/badge/Open%20In-Kaggle-blue?style=for-the-badge&logo=kaggle)](https://kaggle.com/kernels/welcome?src=https://github.com/dnth/rag-datakit/blob/main/nbs/03_filtering-quality-data.ipynb)

This notebook demonstrates how to filter and clean synthetic triplet data to create high-quality training datasets for embedding model fine-tuning. Using semantic similarity metrics, we identify optimal anchor-positive-negative combinations that provide effective learning signals for contrastive training.

## What you'll learn:
- How to compute semantic similarity between text pairs using embeddings
- Filtering strategies for triplet data quality assessment
- Creating hard negatives for more effective contrastive learning
- Splitting filtered data into training and validation sets
- Publishing cleaned datasets to Hugging Face Hub

## Installation

Install the rag-datakit package which includes all necessary dependencies including distilabel, transformers, and dataset utilities. Uncomment the cell below to install if you haven't already.

On Google Colab you might need to uninstall the existing packages due to conflicting versions.

In [ ]:
# !pip uninstall -y transformers torch torchvision

In [ ]:
# !pip install git+https://github.com/dnth/rag-datakit.git

## Import Required Libraries

We begin by importing the essential libraries for data filtering and quality assessment:

- **sentence_transformers**: For loading pre-trained embedding models to compute semantic similarity
- **datasets**: Hugging Face's library for loading and managing synthetic triplet datasets
- **sklearn**: For computing cosine similarity between embeddings

## Load Pre-trained Embedding Model

We'll use the Snowflake Arctic embedding model to compute semantic similarities between text pairs. This model provides high-quality embeddings that help us assess the quality of our triplet data.

In [ ]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, concatenate_datasets
import torch

model_id = "Snowflake/snowflake-arctic-embed-m"  # Use a reasonably good model here

model_retrieval = SentenceTransformer(
    model_id, device="cuda" if torch.cuda.is_available() else "cpu"
)

Now let's load a dataset to filter. I will use an existing dataset that I made some time ago.

## Load and Combine Multiple Datasets

Here we load several synthetic datasets from Hugging Face Hub that were previously generated for retrieval training. We combine multiple dataset configurations (easy/hard variants) to create a comprehensive training corpus with diverse difficulty levels.

In [2]:
dataset_easy = load_dataset("dnth/ssf-synthetic-data-for-retriever-openai", "generate_retrieval_pairs_easy")
dataset_easy_v2 = load_dataset("dnth/ssf-synthetic-data-for-retriever-openai", "generate_retrieval_pairs_easy_v2")
dataset_easy_v3 = load_dataset("dnth/ssf-synthetic-data-for-retriever-openai", "generate_retrieval_pairs_easy_v3")
dataset_hard = load_dataset("dnth/ssf-synthetic-data-for-retriever-openai", "generate_retrieval_pairs_hard")
dataset_hard_v2 = load_dataset("dnth/ssf-synthetic-data-for-retriever-openai", "generate_retrieval_pairs_hard_v2")

dataset = concatenate_datasets([dataset_easy['train'], dataset_hard['train'], 
                                dataset_easy_v2['train'], dataset_easy_v3['train'],
                                dataset_hard_v2['train']])
dataset

Dataset({
    features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
    num_rows: 9425
})

In [3]:
dataset

Dataset({
    features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
    num_rows: 9425
})

In [4]:
dataset = dataset.select_columns(['anchor', 'positive', 'negative'])
dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 9425
})

## Clean Dataset Structure

We simplify the dataset by keeping only the essential triplet columns (anchor, positive, negative) that we need for quality filtering and training. This reduces memory usage and focuses on the core data needed for embedding training.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def get_embeddings(texts):
    vectors = model_retrieval.encode(texts)
    return [vector.tolist() for vector in vectors]


def get_similarities(vector_batch_a, vector_batch_b):
    similarities = []
    for vector_a, vector_b in zip(vector_batch_a, vector_batch_b):
        similarity = cosine_similarity([vector_a], [vector_b])[0][0]
        similarities.append(similarity)
    return similarities

def format_data_retriever(batch):# -&gt; Any:
    batch["anchor-vector"] = get_embeddings(batch["anchor"])
    batch["positive-vector"] = get_embeddings(batch["positive"])
    batch["negative-vector"] = get_embeddings(batch["negative"])    
    batch["similarity-positive-negative"] = get_similarities(batch["positive-vector"], batch["negative-vector"])
    batch["similarity-anchor-positive"] = get_similarities(batch["anchor-vector"], batch["positive-vector"])
    batch["similarity-anchor-negative"] = get_similarities(batch["anchor-vector"], batch["negative-vector"])
    return batch

## Define Quality Assessment Functions

These functions compute semantic similarity scores between different text pairs in our triplets:

- `get_embeddings()`: Converts text to vector representations using our embedding model
- `get_similarities()`: Computes cosine similarity between embedding vectors
- `format_data_retriever()`: Processes batches of triplets to add similarity scores

The similarity scores help us identify high-quality triplets where:
- Anchor-positive pairs have high similarity (good matches)
- Anchor-negative pairs have moderate similarity (hard negatives)
- Positive-negative pairs are sufficiently different

In [6]:
dataset = dataset.map(format_data_retriever, batched=True, batch_size=250)


Map:   0%|          | 0/9425 [00:00<?, ? examples/s]

## Compute Similarity Scores

We process the entire dataset in batches to compute embedding vectors and similarity scores for all triplet combinations. This step may take several minutes depending on dataset size and hardware capabilities.

In [7]:
dataset.to_pandas()

,anchor,positive,negative,anchor-vector,positive-vector,negative-vector,similarity-positive-negative,similarity-anchor-positive,similarity-anchor-negative
0,The Audit Associate/Audit Assistant Associate ...,Audit Assistant role focused on supporting aud...,Junior Financial Analyst responsible for condu...,"[0.06092928722500801, 0.09926585108041763, 0.0...","[-0.005136642139405012, 0.061963580548763275, ...","[-0.01441602036356926, 0.04799262434244156, 0....",0.677419,0.852380,0.597963
1,The Audit Senior Manager/Audit Manager manages...,Audit Manager position responsible for oversee...,Junior Financial Analyst needed to support the...,"[0.057781320065259933, 0.0717240646481514, -0....","[0.0021072286181151867, 0.055933259427547455, ...","[0.004680489655584097, 0.016763579100370407, -...",0.626584,0.804083,0.517771
2,The Audit Partner/Audit Director is a transfor...,Audit Director with strategic leadership skill...,Junior Risk Analyst responsible for assessing ...,"[0.03437988832592964, 0.07796178758144379, -0....","[-0.0442582331597805, 0.02149917371571064, -0....","[-0.012000005692243576, 0.005882208701223135, ...",0.447491,0.629212,0.516452
3,The Audit Senior is expected to team lead vari...,Audit Manager responsible for leading diverse ...,Junior Financial Analyst role focused on condu...,"[0.02032621018588543, 0.06756342202425003, -0....","[-0.03247842192649841, 0.04689886420965195, -0...","[-0.03669586405158043, 0.010867101140320301, -...",0.645973,0.810208,0.619202
4,The Business Valuation Associate/Business Valu...,Business Valuation Analyst role focused on han...,Junior Risk Management Analyst responsible for...,"[0.0326661616563797, 0.050406984984874725, -0....","[-0.004249203950166702, 0.04303452745079994, -...","[-0.011835413053631783, 0.030619123950600624, ...",0.711297,0.838294,0.585513
...,...,...,...,...,...,...,...,...,...
9420,The WSH Manager is responsible for reviewing W...,WSH Manager role focusing on updating and advi...,Junior Safety Coordinator tasked with implemen...,"[-0.013950521126389503, 0.08462849259376526, -...","[-0.023551685735583305, 0.04961637407541275, -...","[0.008779728785157204, 0.04856429994106293, -0...",0.689554,0.904475,0.662994
9421,The WSH Officer is responsible for developing ...,Safety Officer responsible for creating and ov...,Environmental Health Officer tasked with manag...,"[-0.0010848721722140908, 0.050224557518959045,...","[-0.018264105543494225, 0.043748605996370316, ...","[0.039426837116479874, 0.08786812424659729, -0...",0.763903,0.774107,0.697925
9422,The Workplace Safety and Health (WSH) Supervis...,Workplace Safety and Health Supervisor respons...,Environmental Health and Safety (EHS) Coordina...,"[0.020672129467129707, 0.08999118953943253, -0...","[-0.01895511895418167, 0.0747402161359787, -0....","[0.05213961377739906, 0.06548428535461426, -0....",0.673385,0.784742,0.697370
9423,The Lead Workplace Safety and Health (WSH) Aud...,Lead Occupational Health and Safety Auditor ov...,Junior Workplace Safety and Health Coordinator...,"[-0.003643275471404195, 0.04361744970083237, -...","[-0.01498213317245245, 0.0614958256483078, -0....","[-0.00041769916424527764, 0.04934053122997284,...",0.767591,0.787377,0.640114


## Inspect the Enriched Dataset

Let's examine the dataset with the newly computed similarity scores. Each row now includes:
- Original triplet text (anchor, positive, negative)  
- Embedding vectors for each text
- Similarity scores between all pairs (anchor-positive, anchor-negative, positive-negative)

This data will help us filter for high-quality training examples.

## Apply Quality Filtering Criteria

We define filtering criteria to select high-quality triplets for training:

**Filtering Rules:**
- **Anchor-Positive similarity > 0.6**: Ensures positive examples are semantically related to anchors
- **Anchor-Negative similarity 0.2-0.6**: Creates "hard negatives" that are somewhat related but not too similar
- **Positive-Negative similarity < 0.7**: Ensures positive and negative examples are sufficiently distinct

These criteria help create effective contrastive learning examples that improve embedding model performance.

In [8]:
def filter_with_hard_negatives(example):
    anchor_pos = example["similarity-anchor-positive"]
    anchor_neg = example["similarity-anchor-negative"] 
    pos_neg = example["similarity-positive-negative"]
    
    return (
        anchor_pos > 0.6 and  # Good positive
        0.2 < anchor_neg < 0.6 and  # Hard negative range
        pos_neg < 0.7  # Positive and negative are distinct
    )


cleaned_dataset = dataset.filter(filter_with_hard_negatives)

Filter:   0%|          | 0/9425 [00:00<?, ? examples/s]

In [9]:
cleaned_dataset = cleaned_dataset.select_columns(['anchor', 'positive', 'negative'])
cleaned_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 5225
})

## Clean Filtered Dataset

After filtering, we remove the embedding vectors and similarity scores to keep only the essential text data. This reduces storage requirements while preserving the high-quality triplets identified by our filtering process.

In [10]:
train_size = int(0.8 * len(cleaned_dataset))
valid_size = len(cleaned_dataset) - train_size

train_dataset = cleaned_dataset.select(range(train_size))
valid_dataset = cleaned_dataset.select(range(train_size, train_size + valid_size))

## Split into Training and Validation Sets

We create an 80/20 split of our filtered data for training and validation. This ensures we have a proper held-out validation set to monitor model performance during fine-tuning and prevent overfitting.

In [11]:
train_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 4180
})

## Verify Dataset Splits

Let's check the size of our training and validation sets to ensure the split was applied correctly.

In [12]:
valid_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 1045
})

In [13]:
from datasets import DatasetDict

ds = DatasetDict({
    "train": train_dataset,
    "valid": valid_dataset
})

ds

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 4180
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 1045
    })
})

## Create Dataset Dictionary

We package our training and validation splits into a Hugging Face DatasetDict format, which provides a convenient structure for managing both splits together and is the standard format expected by training libraries.

In [14]:
ds.push_to_hub("dnth/ssf-train-valid")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  91%|#########1| 2.72MB / 2.98MB            

...

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  86%|########6 |  650kB /  752kB            

CommitInfo(commit_url='https://huggingface.co/datasets/dnth/ssf-train-valid/commit/591c9372c7dabde6852712f553f8033152f6cdf8', commit_message='Upload dataset', commit_description='', oid='591c9372c7dabde6852712f553f8033152f6cdf8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/dnth/ssf-train-valid', endpoint='https://huggingface.co', repo_type='dataset', repo_id='dnth/ssf-train-valid'), pr_revision=None, pr_num=None)

## Publish to Hugging Face Hub

Finally, we upload our cleaned and filtered dataset to Hugging Face Hub for easy sharing and reuse. This makes the high-quality training data publicly available for the community and provides a versioned, accessible dataset for embedding model fine-tuning.

## Summary

In this notebook, we successfully demonstrated a complete pipeline for filtering and cleaning synthetic triplet data:

1. **Loaded and combined** multiple synthetic datasets from different configurations
2. **Computed semantic similarity scores** using a pre-trained embedding model  
3. **Applied quality filtering criteria** to identify high-quality triplets with hard negatives
4. **Reduced dataset size** from 9,425 to 5,225 examples (55% retention rate)
5. **Split data** into training (4,180 examples) and validation (1,045 examples) sets
6. **Published the cleaned dataset** to Hugging Face Hub for community use

The filtered dataset contains triplets optimized for contrastive learning, with:
- Strong anchor-positive relationships (similarity > 0.6)
- Challenging but learnable negatives (similarity 0.2-0.6) 
- Clear distinction between positives and negatives (similarity < 0.7)

This high-quality dataset is now ready for embedding model fine-tuning and should provide better training signals compared to the original unfiltered data.